Imports

In [425]:
# ==========================================
# IMPORTS
# ==========================================

import sqlite3
import uuid
from datetime import datetime, timedelta
import os

if os.path.exists("notification.db"):
    os.remove("notification.db")

# ==========================================
# SQLITE DATABASE
# ==========================================

connection = sqlite3.connect("notification.db")

cursor = connection.cursor()

cursor.execute("""

CREATE TABLE IF NOT EXISTS notifications(

id TEXT PRIMARY KEY,

user TEXT,

title TEXT,

message TEXT,

status TEXT,

created_time TEXT

)

""")

connection.commit()

print("Notification Database Ready")

Notification Database Ready


Client Layer

This directly represents

User
    ↓
Mobile/Web App

In [426]:
# ==========================================
# CLIENT LAYER
# ==========================================
device = {
    "permission": True,
    "network": True,
    "phone_mode": "Normal",
    "app_state": "Foreground"
}

def client_layer(user, title, message):

    print("\n========== CLIENT LAYER ==========")

    notification = {

        "id": str(uuid.uuid4())[:8],

        "user": user,

        "title": title,

        "message": message,

        "status": "CREATED",

        "retry": 0,

        "time": datetime.now()

    }

    print("User Triggered Notification")

    print("Notification Created")

    return notification

API Layer Matches

API Gateway

↓

Authentication

↓

Notification API

In [427]:
# ==========================================
# API LAYER
# ==========================================

# Device Token Store (Simulation)

device_token_store = {

    "Sakshi": "TOKEN123456"

}

def api_layer(notification):

    print("\n========== API LAYER ==========")

    print("API Gateway Received Request")

    print("Authentication Successful")

    token = device_token_store.get(notification["user"])

    if token:

        print("Device Token Verified")

    else:

        print("Device Token Not Found")

    cursor.execute("""

    INSERT INTO notifications

    VALUES(?,?,?,?,?,?)

    """,(

        notification["id"],

        notification["user"],

        notification["title"],

        notification["message"],

        notification["status"],

        str(notification["time"])

    ))

    connection.commit()

    print("Notification Saved into Database")

    return notification

In [428]:
# ==========================================
# DATABASE 
# ==========================================

def show_database():

    print("\n========== DATABASE ==========")

    cursor.execute("SELECT * FROM notifications")

    rows = cursor.fetchall()

    for row in rows:

        print(row)

Event Streaming

Publish Notification Event

↓

Notification Queue

In [429]:
# ==========================================
# EVENT STREAMING
# ==========================================

notification_queue = []

def event_streaming(notification):

    print("\n========== EVENT STREAMING ==========")

    notification["status"] = "QUEUED"

    notification_queue.append(notification)

    print("Notification Published")

    print("Added to Notification Queue")

Notification Processing layer 
Notification Queue
        │
Notification Worker
        │
Device Status Service
        │
Delivery Decision Engine
        │
Retry Queue / Dead Letter Queue

In [430]:
# ==========================================
# NOTIFICATION PROCESSING
# ==========================================

retry_queue = []

dead_letter_queue = []

# Server Network Status
server_network = True



def notification_processing():

    print("\n========== NOTIFICATION PROCESSING ==========")

    # Queue Empty
    if len(notification_queue) == 0:

        print("Notification Queue Empty")

        return None

    # Notification Worker
    notification = notification_queue.pop(0)

    print("Notification Worker")

    print("Processing Notification :", notification["id"])

    print("\nDelivery Decision Engine")

    # Network Available
    if server_network:

        print("Network Available")

        print("Notification Ready For Delivery")

        return notification

    # Network Not Available
    else:

        print("Network Not Available")

        notification["retry"] += 1

        if notification["retry"] <= 3:

            retry_queue.append(notification)

            print("Moved to Retry Queue")

        else:

            dead_letter_queue.append(notification)

            notification["status"] = "FAILED"

            cursor.execute("""

            UPDATE notifications

            SET status=?

            WHERE id=?

            """, ("FAILED", notification["id"]))

            connection.commit()

            print("Retry Limit Exceeded")

            print("Moved to Dead Letter Queue")

        return None

Push Deliver

Push Gateway

↓

Firebase/APNS

↓

User Device

↓

Notification Display

In [431]:
# ==========================================
# PUSH DELIVERY + USER DEVICE
# ==========================================

# Simulated User Device

def push_delivery(notification):

    print("\n========== PUSH DELIVERY ==========")

    # Permission Check
    if not device["permission"]:
        print("❌ Notification Permission Blocked")
        print("Notification Dropped")
        return notification

    # Network Check
    if not device["network"]:
        print("❌ Network Unavailable")
        print("Added to Retry Queue")
        retry_queue.append(notification)
        return notification

    # App State
    if device["app_state"] == "Foreground":
        print("Application Running In Foreground")

    elif device["app_state"] == "Background":
        print("Application Running In Background")
        print("Displaying System Notification")

    elif device["app_state"] == "Hidden":
        print("Application Hidden")
        print("Notification Stored In Notification Tray")

    # Phone Mode
    if device["phone_mode"] == "Normal":
        print("Phone Mode : Normal")
        print("Playing Notification Sound")

    elif device["phone_mode"] == "Silent":
        print("Phone Mode : Silent")
        print("No Sound")
        print("Showing Silent Notification")

    elif device["phone_mode"] == "Vibrate":
        print("Phone Mode : Vibrate")
        print("Phone Vibrates")
        print("Displaying Notification")

    elif device["phone_mode"] == "Meeting":
        print("Phone Mode : Meeting")
        print("Priority Banner Displayed")
        print("No Sound")

    print("✅ Notification Delivered")

    return notification

Storage Layer
Notification Database

Redis Cache (24 Hours)

Cleanup Scheduler

In [432]:
# ==========================================
# STORAGE LAYER
# ==========================================

history_cache = []

device_token_store = {

    "Sakshi": "TOKEN123"

}


def storage_layer(notification):

    if notification is None:
        return

    print("\n========== STORAGE LAYER ==========")

    # Update SQLite Database

    cursor.execute("""

    UPDATE notifications

    SET status=?

    WHERE id=?

    """,(notification["status"],notification["id"]))

    connection.commit()

    print("Notification Database Updated")

    # Store in Redis Cache

    history_cache.append({

        "notification":notification,

        "expiry":datetime.now()+timedelta(hours=24)

    })

    print("Stored in Redis Cache (24 Hours)")

    print("History Cache Size :",len(history_cache))

    # Device Token Store

    print("Device Token :",device_token_store[notification["user"]])

    def show_history():

      print("\n========== REDIS CACHE ==========")
      show_history()

    if len(history_cache) == 0:

        print("Cache Empty")

        return

    for item in history_cache:

        print("--------------------------------")

        print("ID :", item["notification"]["id"])

        print("Title :", item["notification"]["title"])

        print("Status :", item["notification"]["status"])

        print("Expires :", item["expiry"])

In [433]:
# Cleanup Scheduler
def cleanup_scheduler():

    print("\nRunning Cleanup Scheduler...")

    current_time = datetime.now()

    history_cache[:] = [

        item

        for item in history_cache

        if item["expiry"] > current_time

    ]

    print("Expired Notifications Deleted")

    print("History Cache :",len(history_cache))

In [434]:
# Database Viewer
def show_database():

    print("\n========== DATABASE ==========")

    cursor.execute("SELECT * FROM notifications")

    rows = cursor.fetchall()

    for row in rows:

        print(row)

Main Flow

In [435]:
notification = client_layer(

    "Sakshi",

    "Order Delivered",

    "Your order has arrived."

)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)

storage_layer(notification)

show_database()


========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : 768b0077

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
Application Running In Foreground
Phone Mode : Normal
Playing Notification Sound
✅ Notification Delivered

========== STORAGE LAYER ==========
Notification Database Updated
Stored in Redis Cache (24 Hours)
History Cache Size : 1
Device Token : TOKEN123
--------------------------------
ID : 768b0077
Title : Order Delivered
Status : QUEUED
Expires : 2026-06-23 04:36:53.623734

========== DATABASE ==========
('768b0077', 'Sakshi', 'Order Delivered', 'Your order has 

Scenario 1 (Normal Notification)

In [436]:
print("\n========== SCENARIO 1 : NORMAL ==========")

device["network"] = True
device["permission"] = True
device["phone_mode"] = "Normal"
device["app_state"] = "Foreground"

notification = client_layer(
    "Sakshi",
    "Order Delivered",
    "Your order has arrived."
)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)








========== SCENARIO 1 : NORMAL ==========

========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : b2a167f1

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
Application Running In Foreground
Phone Mode : Normal
Playing Notification Sound
✅ Notification Delivered


Scenario 2 (Network Unavailable)

In [437]:
print("\n========== SCENARIO 2 : NETWORK OFF ==========")

device["network"] = False
device["permission"] = True
device["phone_mode"] = "Normal"
device["app_state"] = "Foreground"

notification = client_layer(
    "Sakshi",
    "Payment Successful",
    "₹500 Credited"
)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)






========== SCENARIO 2 : NETWORK OFF ==========

========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : cf30d6e4

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
❌ Network Unavailable
Added to Retry Queue


Scenario 3 (Silent Mode)

In [438]:
print("\n========== SCENARIO 3 : SILENT MODE ==========")

device["network"] = True
device["permission"] = True
device["phone_mode"] = "Silent"
device["app_state"] = "Foreground"

notification = client_layer(
    "Sakshi",
    "Meeting Reminder",
    "Meeting starts in 10 minutes."
)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)






========== SCENARIO 3 : SILENT MODE ==========

========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : 01037e2b

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
Application Running In Foreground
Phone Mode : Silent
No Sound
Showing Silent Notification
✅ Notification Delivered


Scenario 4 (Vibrate Mode)

In [439]:
print("\n========== SCENARIO 4 : VIBRATE MODE ==========")

device["network"] = True
device["permission"] = True
device["phone_mode"] = "Vibrate"
device["app_state"] = "Foreground"

notification = client_layer(
    "Sakshi",
    "OTP",
    "Your OTP is 123456"
)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)


========== SCENARIO 4 : VIBRATE MODE ==========

========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : 44454d65

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
Application Running In Foreground
Phone Mode : Vibrate
Phone Vibrates
Displaying Notification
✅ Notification Delivered


Scenario 5 (Meeting Mode)

In [440]:
print("\n========== SCENARIO 5 : MEETING MODE ==========")

device["network"] = True
device["permission"] = True
device["phone_mode"] = "Meeting"
device["app_state"] = "Foreground"

notification = client_layer(
    "Sakshi",
    "Emergency Alert",
    "Please check immediately"
)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)


========== SCENARIO 5 : MEETING MODE ==========

========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : 791fe39c

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
Application Running In Foreground
Phone Mode : Meeting
Priority Banner Displayed
No Sound
✅ Notification Delivered


Scenario 6 (Permission Blocked)

In [441]:
print("\n========== SCENARIO 6 : PERMISSION BLOCKED ==========")

device["network"] = True
device["permission"] = False
device["phone_mode"] = "Normal"
device["app_state"] = "Foreground"

notification = client_layer(
    "Sakshi",
    "Friend Request",
    "John sent you a request"
)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)


========== SCENARIO 6 : PERMISSION BLOCKED ==========

========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : 1b8dfbb5

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
❌ Notification Permission Blocked
Notification Dropped


Scenario 7 (App Hidden)

In [442]:
print("\n========== SCENARIO 7 : APP HIDDEN ==========")

device["network"] = True
device["permission"] = True
device["phone_mode"] = "Normal"
device["app_state"] = "Hidden"

notification = client_layer(
    "Sakshi",
    "Bank Alert",
    "₹10,000 Credited"
)

notification = api_layer(notification)

event_streaming(notification)

notification = notification_processing()

notification = push_delivery(notification)


========== SCENARIO 7 : APP HIDDEN ==========

========== CLIENT LAYER ==========
User Triggered Notification
Notification Created

========== API LAYER ==========
API Gateway Received Request
Authentication Successful
Device Token Verified
Notification Saved into Database

========== EVENT STREAMING ==========
Notification Published
Added to Notification Queue

========== NOTIFICATION PROCESSING ==========
Notification Worker
Processing Notification : 9aabaead

Delivery Decision Engine
Network Available
Notification Ready For Delivery

========== PUSH DELIVERY ==========
Application Hidden
Notification Stored In Notification Tray
Phone Mode : Normal
Playing Notification Sound
✅ Notification Delivered


Scenario 8 (History)

In [443]:
print("\n========== HISTORY ==========")

show_database()


========== HISTORY ==========

========== DATABASE ==========
('768b0077', 'Sakshi', 'Order Delivered', 'Your order has arrived.', 'QUEUED', '2026-06-22 04:36:53.621175')
('b2a167f1', 'Sakshi', 'Order Delivered', 'Your order has arrived.', 'CREATED', '2026-06-22 04:36:53.630378')
('cf30d6e4', 'Sakshi', 'Payment Successful', '₹500 Credited', 'CREATED', '2026-06-22 04:36:53.639090')
('01037e2b', 'Sakshi', 'Meeting Reminder', 'Meeting starts in 10 minutes.', 'CREATED', '2026-06-22 04:36:53.646723')
('44454d65', 'Sakshi', 'OTP', 'Your OTP is 123456', 'CREATED', '2026-06-22 04:36:53.653513')
('791fe39c', 'Sakshi', 'Emergency Alert', 'Please check immediately', 'CREATED', '2026-06-22 04:36:53.660202')
('1b8dfbb5', 'Sakshi', 'Friend Request', 'John sent you a request', 'CREATED', '2026-06-22 04:36:53.666800')
('9aabaead', 'Sakshi', 'Bank Alert', '₹10,000 Credited', 'CREATED', '2026-06-22 04:36:53.673623')


Scenario 9 (Redis Cache)

In [424]:
print("\n========== REDIS CACHE ==========")

for item in history_cache:
    print(item)


========== REDIS CACHE ==========


Scenario 10 (Cleanup after 24 Hours)

In [444]:
print("\n========== CLEANUP ==========")

for item in history_cache:
    item["expiry"] = datetime.now() - timedelta(seconds=1)

cleanup_scheduler()

show_database()


========== CLEANUP ==========

Running Cleanup Scheduler...
Expired Notifications Deleted
History Cache : 0

========== DATABASE ==========
('768b0077', 'Sakshi', 'Order Delivered', 'Your order has arrived.', 'QUEUED', '2026-06-22 04:36:53.621175')
('b2a167f1', 'Sakshi', 'Order Delivered', 'Your order has arrived.', 'CREATED', '2026-06-22 04:36:53.630378')
('cf30d6e4', 'Sakshi', 'Payment Successful', '₹500 Credited', 'CREATED', '2026-06-22 04:36:53.639090')
('01037e2b', 'Sakshi', 'Meeting Reminder', 'Meeting starts in 10 minutes.', 'CREATED', '2026-06-22 04:36:53.646723')
('44454d65', 'Sakshi', 'OTP', 'Your OTP is 123456', 'CREATED', '2026-06-22 04:36:53.653513')
('791fe39c', 'Sakshi', 'Emergency Alert', 'Please check immediately', 'CREATED', '2026-06-22 04:36:53.660202')
('1b8dfbb5', 'Sakshi', 'Friend Request', 'John sent you a request', 'CREATED', '2026-06-22 04:36:53.666800')
('9aabaead', 'Sakshi', 'Bank Alert', '₹10,000 Credited', 'CREATED', '2026-06-22 04:36:53.673623')
